In [ ]:
import os
import io
from tqdm import tqdm

from PIL import Image
from datasets import Features, Value, ClassLabel, Dataset, load_from_disk, concatenate_datasets
from datasets import Image as DImage

In [ ]:
anchor_dict = {}
positive_dict = {}
category_dict = {}

labels = ['bag', 'bottom', 'dress', 'hat', 'shoes', 'outer', 'top']
label2id = {
    l: i for i, l in enumerate(labels)
}

pair_id = 0

categories = os.listdir('./pair_images')
for category in categories:
    post_ids = os.listdir(f'./pair_images/{category}')
    for post_id in tqdm(post_ids, desc=category):
        category_id = label2id[category]
        anchor_image_path = f'./pair_images/{category}/{post_id}/anchor.jpg'
        positive_image_path = f'./pair_images/{category}/{post_id}/positive.jpg'

        anchor_dict[pair_id] = anchor_image_path
        positive_dict[pair_id] = positive_image_path
        category_dict[pair_id] = category_id

        pair_id += 1

assert len(anchor_dict) == len(positive_dict) == len(category_dict)
print(f'anchor: {len(anchor_dict)}')
print(f'positive: {len(positive_dict)}')
print(f'category: {len(category_dict)}')

In [ ]:
class_label = ClassLabel(names=labels)
features = Features({
    'pair_id': Value('string'),
    'anchor_image': DImage(decode=True),
    'positive_image': DImage(decode=True),
    'category': class_label,
})

seg = 0
data_list = []
for pair_id in tqdm(anchor_dict.keys()):
    anchor_image_path = anchor_dict[pair_id]
    try:
        anchor_image = Image.open(anchor_image_path).convert('RGB')
    except Exception:
        continue
    anchor_jpeg_buffer = io.BytesIO()
    anchor_image.save(anchor_jpeg_buffer, format='JPEG')
    anchor_jpeg_buffer.seek(0)
    anchor_image = Image.open(anchor_jpeg_buffer)

    positive_image_path = positive_dict[pair_id]
    try:
        positive_image = Image.open(positive_image_path).convert('RGB')
    except Exception:
        continue
    positive_jpeg_buffer = io.BytesIO()
    positive_image.save(positive_jpeg_buffer, format='JPEG')
    positive_jpeg_buffer.seek(0)
    positive_image = Image.open(positive_jpeg_buffer)

    category = category_dict[pair_id]
    data_list.append(
        {
            'pair_id': pair_id,
            'anchor_image': anchor_image,
            'positive_image': positive_image,
            'category': category,
        }
    )

    if len(data_list) == 10000:
        dataset = Dataset.from_list(data_list, features=features)
        dataset.save_to_disk(f'./data_segs/seg_{seg}')
        seg += 1
        data_list.clear()

dataset = Dataset.from_list(data_list, features=features)
dataset.save_to_disk(f'./data_segs/seg_{seg}')
data_list.clear()

In [ ]:
sub_datasets = []
segs = os.listdir('./data_segs')
for seg in segs:
    sub_dataset = load_from_disk(f'./data_segs/{seg}')
    sub_datasets.append(sub_dataset)

dataset = concatenate_datasets(sub_datasets)
dataset = dataset.train_test_split(test_size=0.1, stratify_by_column='category')

dataset.save_to_disk('./anchor_positive_dataset')